In [31]:
#Import Libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [32]:
#Step-2:Load dataset
data = pd.read_csv("/content/sample_data/restaurant_reviews.csv")
print("Dataset Shape:", data.shape)
print("Columns:", data.columns)



Dataset Shape: (110, 19)
Columns: Index(['Restaurant', 'Review', 'Real', 'Reviewer', 'F1-AWL', 'F2-PAU',
       'F3-ANP', 'F4-ASL', 'F5-NCL', 'F6-NWO', 'F7-NVB', 'F8-NAJ', 'F9-NPV',
       'F10-EMO', 'F11-CDV', 'F12-RED', 'F13-LXD', 'F14-NMV', 'F15-NTY'],
      dtype='object')


In [33]:
data.head()

,Restaurant,Review,Real,Reviewer,F1-AWL,F2-PAU,F3-ANP,F4-ASL,F5-NCL,F6-NWO,F7-NVB,F8-NAJ,F9-NPV,F10-EMO,F11-CDV,F12-RED,F13-LXD,F14-NMV,F15-NTY
0,Rose Restaurant,Great food and great atmosphere! The chicken t...,0,3,4.465909,1.000000,14.250000,17.600000,5,88,9,6,0,0.636364,0.972973,9.200000,0.795181,0,0
1,Rose Restaurant,I had heard good things about Rose Restaurant ...,0,3,3.838983,1.500000,10.000000,19.666667,6,118,17,8,1,0.545455,0.976190,11.166667,0.752294,0,0
2,Rose Restaurant,I was driving by rose restaurant one day and d...,0,3,3.377049,1.142857,8.923077,17.428571,7,122,14,5,1,0.375000,0.864865,11.000000,0.675439,0,0
3,Rose Restaurant,Rose Restaurant had the most modern and up-to-...,0,3,4.044118,1.750000,17.500000,17.000000,4,68,6,7,0,0.578947,0.961538,8.750000,0.819672,0,0
4,Gloria Restaurant,Today is the third time I've come to Gloria Re...,0,3,4.071429,1.500000,10.700000,14.000000,6,84,8,8,0,0.777778,0.937500,7.166667,0.786667,0,0


In [34]:
#Define Target and Features

y = data['Real']

# Drop unnecessary columns (if any)
X = data.drop(columns=['Real'])

# Identify numerical feature columns (F1–F15)
feature_cols = [col for col in data.columns if col.startswith('F')]


In [35]:
#Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)

In [36]:
#Preprocessing (Text + Numerical Features)

preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(stop_words='english'), 'Review'),
        ('num', StandardScaler(), feature_cols)
    ]
)

In [37]:
#Create SVM Pipeline
svm_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LinearSVC(class_weight='balanced'))
])

In [38]:
#Train the Model
svm_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('text',
                                                  TfidfVectorizer(stop_words='english'),
                                                  'Review'),
                                                 ('num', StandardScaler(),
                                                  ['F1-AWL', 'F2-PAU', 'F3-ANP',
                                                   'F4-ASL', 'F5-NCL', 'F6-NWO',
                                                   'F7-NVB', 'F8-NAJ', 'F9-NPV',
                                                   'F10-EMO', 'F11-CDV',
                                                   'F12-RED', 'F13-LXD',
                                                   'F14-NMV', 'F15-NTY'])])),
                ('classifier', LinearSVC(class_weight='balanced'))])

In [39]:
#Evaluate the Model
y_pred = svm_model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.8636363636363636

Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.82      0.86        11
           1       0.83      0.91      0.87        11

    accuracy                           0.86        22
   macro avg       0.87      0.86      0.86        22
weighted avg       0.87      0.86      0.86        22


Confusion Matrix:
 [[ 9  2]
 [ 1 10]]


In [40]:
#TEXT-Based SVM
X = data['Review']
y = data['Real']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('svm', LinearSVC(class_weight='balanced'))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.7727272727272727
              precision    recall  f1-score   support

           0       0.71      0.91      0.80        11
           1       0.88      0.64      0.74        11

    accuracy                           0.77        22
   macro avg       0.79      0.77      0.77        22
weighted avg       0.79      0.77      0.77        22



In [41]:
#Test new review
def check_review(text):
    # The 'model' is a Pipeline that includes the TfidfVectorizer.
    # It can directly predict from raw text.
    result = model.predict([text])

    if result[0] == 1:
        print("Real Review")
    else:
        print("Fake Review")


# try examples
check_review("Amazing food and friendly staff. Loved the ambience!")
check_review("Worst restaurant ever. Don't visit. Waste of money.")

Real Review
Fake Review
